# XGBoost — NYC 311 Ticket Closure Prediction
**Target:** `Y = 1` if a service request is closed within 24 hours, else `0`.

This notebook:
1. Replicates the full feature-engineering pipeline from `Features_creation.ipynb`
2. Applies label encoding to categorical features
3. Trains an XGBoost classifier with stratified cross-validation
4. Tunes hyperparameters with `RandomizedSearchCV`
5. Reports accuracy, AUC-ROC, classification report, and feature importances

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, ConfusionMatrixDisplay)
import xgboost as xgb

print(f"XGBoost version : {xgb.__version__}")
print(f"Pandas version  : {pd.__version__}")

## 1. Load Data

In [ ]:
# ── UPDATE THIS PATH ────────────────────────────────────────────────────────
PATH_TRAIN = "Data/train.csv"   # path to your train.csv
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(PATH_TRAIN)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print(f"Shape: {df.shape}")
df.head(3)

## 2. Feature Engineering
Replicating the full pipeline from `Features_creation.ipynb`.

In [ ]:
# ── 2a. Target variable ──────────────────────────────────────────────────────
date_fmt = '%m/%d/%Y %I:%M:%S %p'
df['Created Date'] = pd.to_datetime(df['Created Date'], format=date_fmt, errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'],  format=date_fmt, errors='coerce')
df = df.dropna(subset=['Created Date'])

max_date = max(df['Created Date'].max(), df['Closed Date'].max())
df['Age at Data Dump'] = max_date - df['Created Date']

unknown = df['Closed Date'].isna() & (df['Age at Data Dump'] <= pd.Timedelta(days=1))
print(f"Dropping {unknown.sum()} rows with unknown 24-hr outcome")
df = df[~unknown].copy()

df['Time to Close'] = df['Closed Date'] - df['Created Date']
df['Y'] = (df['Time to Close'] <= pd.Timedelta(days=1)).astype(int)

print(f"\nY distribution:\n{df['Y'].value_counts(normalize=True).mul(100).round(1).to_string()} %")

In [ ]:
# ── 2b. Problem grouping (163 → 9 categories) ────────────────────────────────
def group_problem(text):
    text = str(text).upper()
    if 'NOISE' in text:
        return 'Noise'
    elif any(w in text for w in ['PARKING', 'VEHICLE', 'DRIVEWAY', 'TAXI', 'FOR HIRE']):
        return 'Vehicles & Parking'
    elif any(w in text for w in ['HEAT', 'PAINT', 'DOOR', 'FLOOR', 'ELECTRIC',
                                  'APPLIANCE', 'ELEVATOR', 'MOLD', 'BOILER']):
        return 'Housing & Buildings'
    elif any(w in text for w in ['WATER', 'SEWER', 'PLUMBING', 'LEAK']):
        return 'Water & Plumbing'
    elif any(w in text for w in ['STREET', 'SIDEWALK', 'HIGHWAY', 'TRAFFIC',
                                  'CURB', 'SIGN', 'BRIDGE']):
        return 'Street & Infrastructure'
    elif any(w in text for w in ['SANITARY', 'DIRTY', 'DUMPING', 'COLLECTION',
                                  'DISPOSAL', 'LITTER', 'SWEEPING']):
        return 'Sanitation & Garbage'
    elif any(w in text for w in ['TREE', 'PARK', 'WOOD', 'STUMP', 'PLANT']):
        return 'Trees & Parks'
    elif any(w in text for w in ['ANIMAL', 'RAT', 'RODENT', 'PEST', 'BEE',
                                  'WASP', 'MOSQUITO']):
        return 'Animals & Pests'
    else:
        return 'Other'

problem_col = 'Problem (formerly Complaint Type)' if 'Problem (formerly Complaint Type)' in df.columns else 'Complaint Type'
df['Problem_Grouped'] = df[problem_col].apply(group_problem)
print(df['Problem_Grouped'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# ── 2c. Location grouping (106 → 6 categories) ───────────────────────────────
def group_location(text):
    text = str(text).upper()
    if any(w in text for w in ['STREET', 'SIDEWALK', 'CURB', 'HIGHWAY', 'ROAD',
                                'INTERSECTION', 'CROSSWALK', 'LANE', 'OVERPASS', 'ISLAND']):
        return 'Street/Sidewalk'
    elif any(w in text for w in ['RESIDENT', 'HOUSE', 'APT', 'APARTMENT',
                                  'DWELLING', 'LOFT', 'HOME']):
        return 'Residential'
    elif any(w in text for w in ['COMMERCIAL', 'STORE', 'CLUB', 'BAR', 'RESTAURANT',
                                  'BUSINESS', 'DELI', 'BAKERY', 'FOOD', 'VENDOR',
                                  'CATERING', 'TATTOO', 'RETAIL', 'OFFICE', 'SHOP']):
        return 'Commercial/Business'
    elif any(w in text for w in ['PARK', 'PLAYGROUND', 'GARDEN', 'SUBWAY', 'TERMINAL',
                                  'AIRPORT', 'BUS', 'SCHOOL', 'GOVERNMENT', 'FERRY',
                                  'SHELTER', 'POOL', 'BRIDGE', 'STATION']):
        return 'Public/Transit'
    else:
        return 'Other/Mixed'

df['Location_Grouped'] = df['Location Type'].apply(group_location)
print(df['Location_Grouped'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
# ── 2d. Binary indicators ─────────────────────────────────────────────────────
df['Is_Landmark'] = df['Landmark'].notna().astype(int)
df['Is_Taxi']     = df['Vehicle Type'].notna().astype(int)

# ── 2e. Datetime features ─────────────────────────────────────────────────────
df['Created_Hour']      = df['Created Date'].dt.hour
df['Created_DayOfWeek'] = df['Created Date'].dt.dayofweek
df['Created_Month']     = df['Created Date'].dt.month
df['Is_Weekend']        = df['Created_DayOfWeek'].isin([5, 6]).astype(int)

print("Engineered features created successfully.")

In [ ]:
# ── 2f. Drop unwanted columns ─────────────────────────────────────────────────
features_to_drop = [
    "Unique Key", "Created Date", "Closed Date", "Agency Name",
    "Problem (formerly Complaint Type)", "Problem Detail (formerly Descriptor)",
    "Additional Details", "Location Type", "Incident Address", "Street Name",
    "Cross Street 1", "Cross Street 2", "Intersection Street 1", "Intersection Street 2",
    "Address Type", "City", "Landmark", "Facility Type", "Community Board",
    "Council District", "BBL", "Park Facility Name", "Park Borough", "Vehicle Type",
    "Taxi Company Borough", "Taxi Pick Up Location", "Taxi Pickup Location",
    "Bridge Highway Name", "Bridge Highway Direction", "Bridge Highway Segment",
    "Road Ramp", "X Coordinate (State Plane)", "Y Coordinate (State Plane)",
    "Latitude", "Longitude", "Location", "Age at Data Dump", "Time to Close",
    # Complaint Type may be the old name
    "Complaint Type",
]
df = df.drop(columns=features_to_drop, errors='ignore')

# Move Y to the end for readability
y = df.pop('Y')
df['Y'] = y

print(f"Final columns ({len(df.columns)}):")
print(df.columns.tolist())
df.head(3)

## 3. Encode Categorical Features
XGBoost needs numeric input. We use `LabelEncoder` on remaining string columns.

In [ ]:
X = df.drop(columns=['Y'])
y = df['Y']

cat_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    # Fill NaN with a placeholder string so LabelEncoder doesn't crash
    X[col] = X[col].fillna('__MISSING__')
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

print("\nDtypes after encoding:")
print(X.dtypes)

## 4. Train / Validation Split (stratified)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,}  |  Validation size : {len(X_val):,}")
print(f"Train Y=1  : {y_train.mean():.3f}  |  Val Y=1         : {y_val.mean():.3f}")

## 5. Baseline XGBoost (default params)

In [ ]:
# Class imbalance weight: ~39% negatives, 61% positives
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_base = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=20,
)

xgb_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

y_pred_base  = xgb_base.predict(X_val)
y_proba_base = xgb_base.predict_proba(X_val)[:, 1]

print(f"\n[Baseline] Accuracy : {accuracy_score(y_val, y_pred_base):.4f}")
print(f"[Baseline] AUC-ROC  : {roc_auc_score(y_val, y_proba_base):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_base))

## 6. Hyperparameter Tuning — RandomizedSearchCV
We search over a broad grid with 5-fold stratified CV.

In [ ]:
param_dist = {
    'n_estimators'     : [200, 400, 600, 800],
    'max_depth'        : [3, 4, 5, 6, 7, 8],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'subsample'        : [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight' : [1, 3, 5, 7],
    'gamma'            : [0, 0.1, 0.2, 0.3, 0.5],
    'reg_alpha'        : [0, 0.01, 0.1, 1],       # L1
    'reg_lambda'       : [1, 2, 5, 10],            # L2
}

xgb_cv = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_cv,
    param_distributions=param_dist,
    n_iter=50,                # increase for better coverage (slower)
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    refit=True,
)

random_search.fit(X_train, y_train)

print("\nBest params:")
print(random_search.best_params_)
print(f"\nBest CV AUC-ROC: {random_search.best_score_:.4f}")

## 7. Evaluate Tuned Model

In [ ]:
best_model = random_search.best_estimator_

y_pred_tuned  = best_model.predict(X_val)
y_proba_tuned = best_model.predict_proba(X_val)[:, 1]

print(f"[Tuned] Accuracy : {accuracy_score(y_val, y_pred_tuned):.4f}")
print(f"[Tuned] AUC-ROC  : {roc_auc_score(y_val, y_proba_tuned):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_tuned))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_tuned, ax=ax,
                                         display_labels=['Y=0 (>24h)', 'Y=1 (≤24h)'],
                                         colorbar=False)
ax.set_title('Confusion Matrix — Tuned XGBoost')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 8. Feature Importances

In [ ]:
importances = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Importances — Tuned XGBoost')
ax.set_ylabel('Importance score')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150)
plt.show()

print("\nTop-10 features:")
print(importances.head(10))

## 9. (Optional) Predict on Test Set

In [ ]:
# ── UPDATE THIS PATH ─────────────────────────────────────────────────────────
PATH_TEST = "Data/test.csv"
# ─────────────────────────────────────────────────────────────────────────────

import os
if os.path.exists(PATH_TEST):
    df_test = pd.read_csv(PATH_TEST)
    if "Unnamed: 0" in df_test.columns:
        df_test = df_test.drop(columns=["Unnamed: 0"])

    # ----- Apply the same feature engineering -----
    df_test['Created Date'] = pd.to_datetime(df_test['Created Date'], format=date_fmt, errors='coerce')
    df_test['Closed Date']  = pd.to_datetime(df_test['Closed Date'],  format=date_fmt, errors='coerce')

    df_test['Problem_Grouped']  = df_test[problem_col].apply(group_problem)
    df_test['Location_Grouped'] = df_test['Location Type'].apply(group_location)
    df_test['Is_Landmark']      = df_test['Landmark'].notna().astype(int)
    df_test['Is_Taxi']          = df_test['Vehicle Type'].notna().astype(int)
    df_test['Created_Hour']     = df_test['Created Date'].dt.hour
    df_test['Created_DayOfWeek']= df_test['Created Date'].dt.dayofweek
    df_test['Created_Month']    = df_test['Created Date'].dt.month
    df_test['Is_Weekend']       = df_test['Created_DayOfWeek'].isin([5, 6]).astype(int)

    X_test = df_test.drop(columns=features_to_drop + ['Y'], errors='ignore')

    for col in cat_cols:
        X_test[col] = X_test[col].fillna('__MISSING__')
        # Map unseen labels to the __MISSING__ class
        known = set(le_dict[col].classes_)
        X_test[col] = X_test[col].apply(lambda v: v if v in known else '__MISSING__')
        X_test[col] = le_dict[col].transform(X_test[col])

    test_proba = best_model.predict_proba(X_test)[:, 1]
    test_pred  = best_model.predict(X_test)

    submission = pd.DataFrame({'Y_pred': test_pred, 'Y_proba': test_proba})
    submission.to_csv('xgb_submission.csv', index=False)
    print(f"Saved {len(submission):,} predictions → xgb_submission.csv")
    print(submission.head())
else:
    print(f"Test file not found at '{PATH_TEST}' — skipping.")